# 01 - Runnables and Chains
Understaing the core LangChain constructs - _Runnables_ and _Chains_.

In [1]:
import random
from operator import itemgetter
from rich.console import Console
import langchain
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [2]:
console = Console()
console.print(f"[red]Using Langchain {langchain.__version__}[/red]")

Using Langchain 1.2.14

The base class for all LangChain classes is `Runnable`, which has methods such as `invoke()`,`stream()` and `batch()` apart from other methods.

A **chain** is formed by sequencing `Runnables` one after another, and that itelf becomes a `Runnable` - hence you can call `invoke()` on a chain.

A `RunnableLambda` is a class that makes any Python function/lambda a `Runnable` object - just wrap the Python function/lambda like this:

In [3]:
# a regular Python function to convert Celsius to Fahrenheit
def c2f(celsius: float) -> float:
    # f = (c * 9/5) + 32
    return celsius * (9.0 / 5.0) + 32.0


# converted to a RunnableLambda
conv = RunnableLambda(c2f)
# now you can invoke the runnable!
c = 45.35
console.print(f"[blue]{c:.2f} Celsius  = {conv.invoke(c):.2f} Fahrenheit[/blue]")

# or with a simple lambda function
conv_lambda = RunnableLambda(lambda c: c * (9.0 / 5.0) + 32.0)
console.print(
    f"[green]{c:.2f} Celsius  = {conv_lambda.invoke(c):.2f} Fahrenheit[/green]"
)

45.35 Celsius  = 113.63 Fahrenheit

45.35 Celsius  = 113.63 Fahrenheit

You can chain runnables/runnable-lambdas together into a chain like this:

In [4]:
def get_random_celsius() -> float:
    c = random.uniform(-100.0, 100.0)
    console.print(f"[red]Generated celsius: {c:.2f}[/red]")
    return c

In [5]:
temp_conv_chain = (
    # have used the funny lambda syntax for the first RunnableLambda as
    # get_random_celsius() does not take a parameter, but invoke() requires at least 1
    RunnableLambda(lambda _: get_random_celsius())
    | RunnableLambda(c2f)
)

# now we invoke - note: invoke() REQUIRES at least 1 param
response = temp_conv_chain.invoke(None)
console.print(f"[blue]Random Celsius converted to Fahrenheit: {response:.2f}[/blue]")

Generated celsius: 63.26

Random Celsius converted to Fahrenheit: 145.86

You can also run chains in _parallel_ as shown below:

In [6]:
def fake_llm(x: int) -> str:
    return f"Fake LLM says: {x}^2 = {x**2}"


# you can also run chains in parallel
parallel_chain = RunnableParallel(
    step1=temp_conv_chain, step2=RunnableLambda(fake_llm)  # same chain as above
)
# NOTE: the 7 here goes to 2nd chain!
response = parallel_chain.invoke(7)
console.print(response)

Generated celsius: 66.24

{'step1': 151.22454304734617, 'step2': 'Fake LLM says: 7^2 = 49'}

Now in the previous `RunnableParallel` example, the first chain does **not** require a parameter. But what if it did? How will that work? Let's make the changes below:

In [7]:
# Modified to accept a parameter
def get_random_celsius(modifier: float) -> float:
    # Let's say it uses the parameter to bias the random number
    c = random.uniform(-100.0, 100.0) + modifier
    console.print(f"[red]Generated celsius with modifier: {c:.2f}[/red]")
    return c

No changes to `fake_llm()`...

In [8]:
# 1. Update the first chain to look for its specific key from the input dict
temp_conv_chain = (
    itemgetter("celsius_modifier")
    | RunnableLambda(get_random_celsius)
    | RunnableLambda(c2f)
)

# 2. Update the parallel chain so step2 also pulls its specific key
parallel_chain = RunnableParallel(
    step1=temp_conv_chain,
    step2=itemgetter("llm_input") | RunnableLambda(fake_llm),
)

# 3. Invoke with a dictionary providing parameters for both!
response = parallel_chain.invoke({"celsius_modifier": 10.5, "llm_input": 7})

console.print(response)

Generated celsius with modifier: -66.98

{'step1': -88.55956681634278, 'step2': 'Fake LLM says: 7^2 = 49'}